## Imports and Constants

In [ ]:
import pynq
import numpy as np
import json
import os

# Import the driver and utilities from the local pynq_driver.py file
# This assumes pynq_driver.py is in the same directory as this notebook
from pynq_driver import DeepSoCFlowPYNQ, unpack_bytes_into_words

# --- Define Paths ---
# Assumes the notebook is in a directory that contains cgra4ml.bit, pynq_driver.py, y_exp.txt, and the 'vectors' folder.
NOTEBOOK_DIR = os.getcwd() 
BITSTREAM_PATH = os.path.join(NOTEBOOK_DIR, 'cgra4ml.bit')
VECTORS_DIR = os.path.join(NOTEBOOK_DIR, 'vectors')
CONFIG_PATH = os.path.join(VECTORS_DIR, 'config.json')
WBX_PATH = os.path.join(VECTORS_DIR, 'wbx.bin')
Y_EXP_PATH = os.path.join(NOTEBOOK_DIR, 'y_exp.txt') 

# --- Check that files exist before proceeding ---
assert os.path.exists(BITSTREAM_PATH), f"Bitstream not found at {BITSTREAM_PATH}"
assert os.path.exists(CONFIG_PATH), f"Config file not found at {CONFIG_PATH}"
assert os.path.exists(WBX_PATH), f"WBX file not found at {WBX_PATH}"
assert os.path.exists(Y_EXP_PATH), f"Expected output file not found at {Y_EXP_PATH}"

print("Setup complete. All necessary files found.")


Setup complete. All necessary files found.


## Load the Overlay

In [ ]:
print("Loading overlay...")
overlay = pynq.Overlay(BITSTREAM_PATH)
print("Overlay loaded successfully.")

overlay.ip_dict


Loading overlay...


Overlay loaded successfully.


{'axi_cgra4ml_0': {'type': 'xilinx.com:module_ref:axi_cgra4ml:1.0',
  'mem_id': 's_axil',
  'memtype': 'REGISTER',
  'gpio': {},
  'interrupts': {},
  'parameters': {'ROWS': '4',
   'COLS': '8',
   'X_BITS': '4',
   'K_BITS': '4',
   'Y_BITS': '20',
   'Y_OUT_BITS': '32',
   'M_DATA_WIDTH_HF_CONV': '640',
   'M_DATA_WIDTH_HF_CONV_DW': '80',
   'AXI_WIDTH': '64',
   'AXI_ID_WIDTH': '6',
   'AXI_STRB_WIDTH': '8',
   'AXI_MAX_BURST_LEN': '16',
   'AXI_ADDR_WIDTH': '32',
   'AXIL_WIDTH': '32',
   'AXIL_ADDR_WIDTH': '32',
   'STRB_WIDTH': '4',
   'W_BPT': '32',
   'Component_Name': 'design_1_axi_cgra4ml_0_0',
   'EDK_IPTYPE': 'PERIPHERAL',
   'C_BASEADDR': '0x43C00000',
   'C_HIGHADDR': '0x43C00FFF',
   'DATA_WIDTH': '64',
   'PROTOCOL': 'AXI4',
   'FREQ_HZ': '100000000',
   'ID_WIDTH': '6',
   'ADDR_WIDTH': '32',
   'AWUSER_WIDTH': '0',
   'ARUSER_WIDTH': '0',
   'WUSER_WIDTH': '0',
   'RUSER_WIDTH': '0',
   'BUSER_WIDTH': '0',
   'READ_WRITE_MODE': 'WRITE_ONLY',
   'HAS_BURST': '1',
   'H

In [3]:
# See all the IPs and hierarchies that PYNQ has detected.
# overlay?

## Load Data and Allocate Buffers

In [ ]:
print("--- Testing Driver Initialization and Memory Allocation ---")

ACCELERATOR_IP_NAME = 'axi_cgra4ml_0'

driver = DeepSoCFlowPYNQ(overlay, config_path=CONFIG_PATH, accelerator_ip_name=ACCELERATOR_IP_NAME)

print("\n--- Driver initialization and memory allocation test complete! ---")

--- Testing Driver Initialization and Memory Allocation ---
Loading configuration from vectors/config.json...
Allocating memory buffers...
Memory allocation complete.

--- Driver initialization and memory allocation test complete! ---


## Setup the model 

In [ ]:
print("\n--- Testing Model Setup ---")
driver.model_setup(wbx_path=WBX_PATH)
print("\n--- Model setup test complete! ---")


--- Testing Model Setup ---

Setting up model from vectors/wbx.bin...
Data copy complete.
Pre-loading all bundle parameters...
Parameter loading complete.
Register configuration complete.
Model setup finished.

--- Model setup test complete! ---


## Model Run

In [ ]:
print("--- Verifying Loaded Input Data ---")

# Define paths to golden data files
# These files are expected to be in the 'vectors' subdirectory
W_GOLDEN_PATH = os.path.join(VECTORS_DIR, '0_0_0_w.txt')
X_GOLDEN_PATH = os.path.join(VECTORS_DIR, '0_0_x.txt') 

# --- 1. Verify 'w' (weights) buffer ---
try:
    w_buffer_bytes = driver.mem['w'].tobytes()
    w_buffer_unpacked = unpack_bytes_into_words(w_buffer_bytes, bits=(1 << driver.defines['W_BITS_L2']))
    w_golden = np.loadtxt(W_GOLDEN_PATH, dtype=np.int8)
    w_matches = np.array_equal(w_buffer_unpacked[:w_golden.size], w_golden)
    print(f"'w' buffer content matches golden file: {w_matches}")
    if not w_matches:
        print("    First 16 values in 'w' buffer:", w_buffer_unpacked[:16])
        print("    First 16 values in golden 'w':", w_golden[:16])
except Exception as e:
    print(f"Error verifying 'w' buffer: {e}")


# --- 2. Verify 'b' (biases) buffer ---
# The biases are simple int32_t values, no complex packing.
try:
    b_buffer = driver.mem['b']
    with open(WBX_PATH, 'rb') as f:
        f.seek(driver.mem['w'].nbytes) # Seek past the weights
        b_golden_bytes = f.read(driver.mem['b'].nbytes)
        b_golden = np.frombuffer(b_golden_bytes, dtype=driver.mem['b'].dtype)
    b_matches = np.array_equal(b_buffer, b_golden)
    print(f"'b' buffer content matches golden file: {b_matches}")
    if not b_matches:
        print("    First 16 values in 'b' buffer:", b_buffer[:16])
        print("    First 16 values in golden 'b':", b_golden[:16])
except Exception as e:
    print(f"Error verifying 'b' buffer: {e}")


# --- 3. Verify 'x' (input) buffer ---
try:
    x_buffer_bytes = driver.mem['x'].tobytes()
    x_buffer_unpacked = unpack_bytes_into_words(x_buffer_bytes, bits=(1 << driver.defines['X_BITS_L2']))
    x_golden = np.loadtxt(X_GOLDEN_PATH, dtype=np.int8)
    x_matches = np.array_equal(x_buffer_unpacked[:x_golden.size], x_golden)
    print(f"'x' buffer content matches golden file: {x_matches}")
    if not x_matches:
        print("    First 16 values in 'x' buffer:", x_buffer_unpacked[:16])
        print("    First 16 values in golden 'x':", x_golden[:16])
except Exception as e:
    print(f"Error verifying 'x' buffer: {e}")


print("--- Input Data Verification Complete ---")



--- Testing Model Run ---

--- Starting Model Run ---
--- Processing Bundle 0 ---
  Tile 0, Pass 0
    Processing tile 0 on CPU for pass 0...
  Tile 1, Pass 0
    Processing tile 1 on CPU for pass 0...
  Tile 2, Pass 0
    Processing tile 2 on CPU for pass 0...
  Tile 3, Pass 0
    Processing tile 3 on CPU for pass 0...
  Tile 4, Pass 0
    Processing tile 4 on CPU for pass 0...
  Tile 5, Pass 0
    Processing tile 5 on CPU for pass 0...
  Tile 6, Pass 0
    Processing tile 6 on CPU for pass 0...
--- Processing Bundle 1 ---
  Tile 0, Pass 0
    Processing tile 0 on CPU for pass 0...
  Tile 1, Pass 0
    Processing tile 1 on CPU for pass 0...
--- Processing Bundle 2 ---
  Tile 0, Pass 0
    Processing tile 0 on CPU for pass 0...
  Tile 1, Pass 0
    Processing tile 1 on CPU for pass 0...
--- Processing Bundle 3 ---
  Tile 0, Pass 0
    Processing tile 0 on CPU for pass 0...
  Tile 1, Pass 0
    Processing tile 1 on CPU for pass 0...
--- Processing Bundle 4 ---
  Tile 0, Pass 0
    Proc

In [ ]:
print("\n--- Testing Model Run ---")

# Run the model. This will use the input data loaded from wbx.bin
output = driver.model_run()

print("\n--- Model run test complete! ---")
print(f"Output shape: {output.shape}")
print(f"Output dtype: {output.dtype}")
print(f"First 10 output values:\n{output[:10]}")

print("\n--- Comparing PYNQ Output with Expected Output ---")
# We already know it's float32 from your y_exp.txt, but for robustness:
last_bundle = driver.bundles[-1]
output_dtype = np.float32 if last_bundle['is_softmax'] else np.int32

expected_output = np.loadtxt(Y_EXP_PATH, dtype=output_dtype)

print(f"Expected output shape: {expected_output.shape}")
print(f"Expected output dtype: {expected_output.dtype}")
print(f"First 10 expected values:\n{expected_output[:10]}")

# Compare the actual output from the PYNQ driver with the expected output
is_close = np.allclose(output, expected_output, atol=1e-2, rtol=1e-2) # Tolerances may need tuning
print(f"\nOutputs are numerically close: {is_close}")

if not is_close:
    print("\nDifferences (first 20 values where they differ or beyond tolerance):")
    # Find indices where values are NOT close
    diff_idx = np.where(~np.isclose(output, expected_output, atol=1e-2, rtol=1e-2))[0]
    if diff_idx.size > 0:
        # This check prevents the IndexError if output is smaller than expected
        max_len = max(len(output), len(expected_output))
        for i in range(max_len):
            if i >= 20: break # Limit to first 20 diffs
            pynq_val = output[i] if i < len(output) else "N/A (out of bounds)"
            exp_val = expected_output[i] if i < len(expected_output) else "N/A (out of bounds)"
            
            if not np.isclose(pynq_val if isinstance(pynq_val, (int, float)) else 0, 
                              exp_val if isinstance(exp_val, (int, float)) else 0, 
                              atol=1e-2, rtol=1e-2):
                diff = abs(pynq_val - exp_val) if isinstance(pynq_val, (int, float)) and isinstance(exp_val, (int, float)) else "N/A"
                print(f"  Index {i}: PYNQ={pynq_val}, Expected={exp_val}, Diff={diff}")
    else:
        print("No significant differences found beyond tolerance, but np.allclose returned False.")

print("\n--- Comparison Complete ---")



## Cleanup

In [ ]:
print("\nCleaning up resources...")
# The try/except blocks handle cases where the notebook is partially run
try:
    if 'driver' in locals() and driver is not None:
        del driver
except NameError:
    pass

try:
    if 'overlay' in locals() and overlay is not None:
        overlay.free()
except NameError:
    pass

print("Cleanup complete.")



Cleaning up resources...

Releasing memory buffers.
Cleanup complete.
